# Orchestrate a multi-step Glue workflow with job bookmarks to process only new arrivals

The payments file feed bumped to hourly. The single Glue job from last quarter now reprocesses the entire backlog on every run, takes 40 minutes, and produces duplicate rows in the curated layer. You have one session to rebuild the pipeline as a Glue workflow with three sequential steps (crawl, transform, load) and turn on Glue 4.0 job bookmarks on the transform step so each run processes only the files added since the last successful execution.

Glue 4.0 PySpark, one workflow with three connected nodes, three trigger objects to chain them. No Step Functions, no EventBridge. The DEA-C01 exam treats Glue workflows + bookmarks as the canonical way to express an incremental ETL pipeline; Lab 04 (`glue-etl-transforms`) built the single ETL job, Lab 07 extends it.

The four deliverables map to four checkpoints:
1. Workflow exists with three nodes (crawler, transform job, load job) connected by one on-demand trigger and two conditional triggers that fire on `SUCCEEDED`.
2. First workflow run completes; transform job processed the seeded staging files; curated/ now has at least one Parquet partition.
3. Second workflow run with no new files processes zero new transforms (the bookmark held); curated/ object count is unchanged between run 1 and run 2.
4. Adding one new CSV to staging/ and running the workflow a third time produces exactly one new partition with no duplicate rows in curated/.

**Time.** About 60 minutes hands-on. Most of the wall-clock time is the Glue 10-minute minimum on each workflow run (three runs in this lab), not your typing. Budget up to 90 minutes if you hit a script bug and need a fourth run.

**Cost.** This lab costs about 30 to 75 cents. Glue ETL bills at $0.44 per DPU-hour with a 10-minute minimum, and the lab runs the workflow three times to prove the bookmark works (one seed run, one no-new-files run, one new-file run); each run is roughly 15 cents at 2 G.1X workers. Glue crawler runs are 2 DPU, 1-minute minimum, about 1.5 cents apiece. The cleanup cell deletes every job, trigger, workflow, crawler, role, database, and bucket at the end so the meter stops the moment you finish.

This lab maps to AWS DEA-C01 Domain 1: Data Ingestion and Transformation (34% of exam weight).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import csv
import getpass
import io
import json
import random
import re
import time
from datetime import datetime, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "glue-workflows-and-bookmarks"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[6].slug exactly
REGION = "us-east-1"  # all DEA-C01 labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource names. Bucket name appends the account ID for global uniqueness.
ROLE_NAME = f"labrun-{LAB_ID}-job-role"
INLINE_POLICY_NAME = f"labrun-{LAB_ID}-job-inline"
DATABASE_NAME = f"labrun_{LAB_ID.replace(chr(45), chr(95))}_db"
CRAWLER_NAME = f"labrun-{LAB_ID}-crawler"
TRANSFORM_JOB_NAME = f"labrun-{LAB_ID}-transform"
LOAD_JOB_NAME = f"labrun-{LAB_ID}-load"
WORKFLOW_NAME = f"labrun-{LAB_ID}-workflow"
START_TRIGGER_NAME = f"labrun-{LAB_ID}-start"
TRANSFORM_TRIGGER_NAME = f"labrun-{LAB_ID}-after-crawler"
LOAD_TRIGGER_NAME = f"labrun-{LAB_ID}-after-transform"
BUCKET_NAME = None  # resolved after STS once the account ID is known

# Seed config. Deterministic so Checkpoint 4 can compute the expected row
# delta from the new CSV without coupling to Checkpoint 1.
SEED = 73
ROWS_PER_FILE = 80
NEW_FILE_ROWS = 30

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"All DEA-C01 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first Glue call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that we know the account ID. S3 bucket names
# must be globally unique.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# The manifest is module-level and in strict reverse-creation order. Lab 07
# has no critical (hourly-billed) resources. Glue jobs, crawlers, and
# workflows bill only while running and the lab caps each run at the 10-min
# Glue minimum on the ETL jobs and 1-minute minimum on the crawler.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist (not just print a warning).
#
# Note: labrun-checks 0.3.0 ships AWS adapters for s3_bucket, iam_role,
# glue_database, and glue_table. It does NOT yet support glue_job,
# glue_crawler, glue_workflow, or glue_trigger. A _LabAdapter subclass
# extending AwsCleanupAdapter is defined in the cleanup cell at the bottom
# of this notebook and passed to run_cleanup. The new resource types are
# still declared declaratively here so the canonical summary, atexit
# handler, and orphan scan all see them.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="glue_trigger",
        id=LOAD_TRIGGER_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-trigger --name {LOAD_TRIGGER_NAME}",
    ),
    CleanupResource(
        type="glue_trigger",
        id=TRANSFORM_TRIGGER_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-trigger --name {TRANSFORM_TRIGGER_NAME}",
    ),
    CleanupResource(
        type="glue_trigger",
        id=START_TRIGGER_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-trigger --name {START_TRIGGER_NAME}",
    ),
    CleanupResource(
        type="glue_workflow",
        id=WORKFLOW_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-workflow --name {WORKFLOW_NAME}",
    ),
    CleanupResource(
        type="glue_job",
        id=LOAD_JOB_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {LOAD_JOB_NAME}",
    ),
    CleanupResource(
        type="glue_job",
        id=TRANSFORM_JOB_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {TRANSFORM_JOB_NAME}",
    ),
    CleanupResource(
        type="glue_crawler",
        id=CRAWLER_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-crawler --name {CRAWLER_NAME}",
    ),
    CleanupResource(
        type="glue_database",
        id=DATABASE_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {DATABASE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this is the
    safety net for kernel crashes. Errors are swallowed because atexit
    handlers must not raise.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter()) if "_LabAdapter" in globals() else run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before creating
    any new resources.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources or")
    print("collide on the workflow, trigger, and job names. Run the cleanup")
    print("cell at the bottom of this notebook first, or remove them manually")
    print("with the AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Build the workflow, its three job nodes, and the triggers that chain them

This is the biggest cell of the lab by line count. Six pieces of plumbing to land:

1. **S3 bucket** with the four prefixes the workflow needs: `staging/` (raw input), `scripts/` (PySpark code for both jobs), `curated/` (Parquet output), `temp/` (Glue scratch).
2. **IAM role** named `labrun-glue-workflows-and-bookmarks-job-role` used by the crawler and both ETL jobs. Glue trust policy, managed `AWSGlueServiceRole` attached, plus an inline policy granting S3 actions on the lab bucket and `/*` ARN.
3. **Glue Data Catalog database** named `labrun_glue_workflows_and_bookmarks_db`. The crawler writes the staging-table schema into this database; the transform job reads from it via `from_catalog`.
4. **Crawler** over `s3://{bucket}/staging/`. The first workflow run kicks the crawler so the staging table appears in the catalog before the transform job runs.
5. **Two Glue ETL jobs**. The transform job ships with `--job-bookmark-option: job-bookmark-enable` and a stable `transformation_ctx` so the bookmark state persists across runs. The load job is bookmarks-disabled (idempotent overwrite into a marker file).
6. **One workflow + three triggers**. `start-trigger` is `ON_DEMAND` and fires the crawler. `after-crawler` is `CONDITIONAL` on the crawler reaching `SUCCEEDED` and fires the transform job. `after-transform` is `CONDITIONAL` on the transform job reaching `SUCCEEDED` and fires the load job. Both conditional triggers must include `StartOnCreation=True` so the workflow controller knows to evaluate them.

After creating the role, sleep 15 seconds before the next cell. IAM role propagation is asynchronous and Glue validates the role at job-create time. Skipping the sleep is the most common false-FAIL on Checkpoint 1.

In [ ]:
# NBVAL_SKIP
# Task 1: Build the full workflow scaffold. Bucket, role, database, crawler,
# two ETL jobs (transform with bookmarks enabled, load without), workflow,
# and three triggers (one ON_DEMAND start, two CONDITIONAL chained on SUCCEEDED).

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Create the bucket. us-east-1 rejects LocationConstraint; other regions
# require it. All DEA-C01 labs run in us-east-1, so the simple call works.
# YOUR CODE: Create the S3 bucket with s3.create_bucket(Bucket=BUCKET_NAME).

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# PySpark script for the transform job. Two things make this a bookmarked
# job rather than a plain ETL job:
#   1. The script calls Job(glueContext) + job.init(...) + job.commit().
#      The Glue job framework persists bookmark state on commit().
#   2. The source DynamicFrame is created with a stable transformation_ctx
#      ("transform_orders_v1"). The bookmark state is keyed on this string;
#      change it and the bookmark is effectively reset.
# The script also prints the read-side row count and the bookmark state to
# stdout so a CloudWatch Logs read can confirm the bookmark is working.
TRANSFORM_SCRIPT = """
import sys
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql.functions import col, month, to_timestamp, year

args = getResolvedOptions(sys.argv, ["JOB_NAME", "BUCKET_NAME", "DATABASE_NAME", "STAGING_TABLE"])
sc = SparkContext.getOrCreate()
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session
job = Job(glue_ctx)
job.init(args["JOB_NAME"], args)

bucket = args["BUCKET_NAME"]
database = args["DATABASE_NAME"]
table = args["STAGING_TABLE"]
curated_path = "s3://" + bucket + "/curated/"

# from_catalog with a stable transformation_ctx is what makes the bookmark
# track files-already-processed across workflow runs.
dyf = glue_ctx.create_dynamic_frame.from_catalog(
    database=database,
    table_name=table,
    transformation_ctx="transform_orders_v1",
)
df = dyf.toDF()

rows_read = df.count()
print("LABRUN_BOOKMARK_OBSERVATION rows_read=" + str(rows_read))

if rows_read == 0:
    print("LABRUN_BOOKMARK_OBSERVATION no_new_files_since_last_bookmark")
    job.commit()
    sys.exit(0)

df = df.withColumn(
    "created_at_ts",
    to_timestamp(col("created_at"), "yyyy-MM-dd\'T\'HH:mm:ss\'Z\'"),
)
df = df.withColumn("year", year(col("created_at_ts")))
df = df.withColumn("month", month(col("created_at_ts")))
df_out = df.drop("created_at_ts").dropDuplicates(["order_id"])

df_out.write.mode("append").partitionBy("year", "month").parquet(curated_path)

job.commit()
""".lstrip()

# PySpark script for the load job. Idempotent: writes a single
# 'last-loaded' marker JSON into curated-meta/. No bookmark, no dedup,
# no partition write. The marker proves the workflow chain reached the
# load step on each run.
LOAD_SCRIPT = """
import sys
import time
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext

args = getResolvedOptions(sys.argv, ["JOB_NAME", "BUCKET_NAME"])
sc = SparkContext.getOrCreate()
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session
job = Job(glue_ctx)
job.init(args["JOB_NAME"], args)

bucket = args["BUCKET_NAME"]
marker_path = "s3://" + bucket + "/curated-meta/load-marker.txt"

marker_rdd = sc.parallelize(["loaded_at_epoch=" + str(int(time.time()))])
marker_rdd.coalesce(1).saveAsTextFile("s3://" + bucket + "/curated-meta/run-" + args["JOB_NAME"] + "-" + str(int(time.time())))
print("LABRUN_LOAD_STEP marker_written=" + marker_path)

job.commit()
""".lstrip()

TRANSFORM_SCRIPT_KEY = "scripts/transform.py"
LOAD_SCRIPT_KEY = "scripts/load.py"
transform_script_uri = f"s3://{BUCKET_NAME}/{TRANSFORM_SCRIPT_KEY}"
load_script_uri = f"s3://{BUCKET_NAME}/{LOAD_SCRIPT_KEY}"

# YOUR CODE: Upload both PySpark scripts to S3 with
#   s3.put_object(Bucket=BUCKET_NAME, Key=TRANSFORM_SCRIPT_KEY, Body=TRANSFORM_SCRIPT.encode("utf-8"))
#   s3.put_object(Bucket=BUCKET_NAME, Key=LOAD_SCRIPT_KEY, Body=LOAD_SCRIPT.encode("utf-8"))

print(f"Transform script uploaded: {transform_script_uri}")
print(f"Load script uploaded: {load_script_uri}")

# Build the Glue trust policy. Principal is the Glue service.
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "glue.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

# YOUR CODE: Create the IAM role with iam.create_role(
#   RoleName=ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(trust_policy),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).

# Attach the AWS-managed AWSGlueServiceRole policy. Glue ETL jobs and
# crawlers need it for CloudWatch Logs + Glue catalog access on top of
# your inline S3 grants.
iam.attach_role_policy(
    RoleName=ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole",
)

# Inline policy: S3 actions on the lab bucket and its /* ARN.
bucket_arn = f"arn:aws:s3:::{BUCKET_NAME}"
inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:PutObject",
                "s3:DeleteObject",
                "s3:ListBucket",
            ],
            "Resource": [bucket_arn, f"{bucket_arn}/*"],
        }
    ],
}

# YOUR CODE: Attach the inline policy with iam.put_role_policy(
#   RoleName=ROLE_NAME,
#   PolicyName=INLINE_POLICY_NAME,
#   PolicyDocument=json.dumps(inline_policy),
# ).

print(f"Role created: {ROLE_NAME}")
print("Managed policy AWSGlueServiceRole attached.")
print("Inline S3 policy attached.")

# Glue database for the staging table the crawler creates and the
# transform job reads via from_catalog.
try:
    glue.create_database(
        DatabaseInput={
            "Name": DATABASE_NAME,
            "Description": f"Workflow catalog for {LAB_ID}",
        },
    )
    glue.tag_resource(
        ResourceArn=f"arn:aws:glue:{REGION}:{ACCOUNT_ID}:database/{DATABASE_NAME}",
        TagsToAdd={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    print(f"Glue database created and tagged: {DATABASE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue database {DATABASE_NAME} already exists, reusing.")
    else:
        raise

# IAM role propagation. Glue validates the role at job-create AND
# crawler-create time. Skipping this sleep is the most common false-FAIL
# on Checkpoint 1.
print("Your IAM role is stuck in traffic, give it 15 seconds...")
time.sleep(15)
print("Role is ready.")

role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
STAGING_TABLE_NAME = "staging"  # crawler writes this name (prefix-derived)

# Crawler over staging/. Schema discovery target for the transform job.
# YOUR CODE: Create the crawler with glue.create_crawler(
#   Name=CRAWLER_NAME,
#   Role=role_arn,
#   DatabaseName=DATABASE_NAME,
#   Targets={"S3Targets": [{"Path": f"s3://{BUCKET_NAME}/staging/"}]},
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).

print(f"Crawler created: {CRAWLER_NAME}")

# Transform job. job-bookmark-enable is the knob that turns this from a
# full-rescan job into a bookmarked-incremental job; the PySpark script
# above already uses a stable transformation_ctx for from_catalog.
transform_default_args = {
    "--BUCKET_NAME": BUCKET_NAME,
    "--DATABASE_NAME": DATABASE_NAME,
    "--STAGING_TABLE": STAGING_TABLE_NAME,
    "--job-bookmark-option": "job-bookmark-enable",
    "--TempDir": f"s3://{BUCKET_NAME}/temp/",
    "--enable-metrics": "true",
}

# YOUR CODE: Create the transform job with glue.create_job(
#   Name=TRANSFORM_JOB_NAME,
#   Role=role_arn,
#   Command={"Name": "glueetl", "ScriptLocation": transform_script_uri, "PythonVersion": "3"},
#   DefaultArguments=transform_default_args,
#   GlueVersion="4.0",
#   WorkerType="G.1X",
#   NumberOfWorkers=2,
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).

print(f"Transform job created with bookmarks ENABLED: {TRANSFORM_JOB_NAME}")

# Load job. Bookmarks DISABLED on this step on purpose; the load script
# writes a marker per run and is intentionally idempotent.
load_default_args = {
    "--BUCKET_NAME": BUCKET_NAME,
    "--job-bookmark-option": "job-bookmark-disable",
    "--TempDir": f"s3://{BUCKET_NAME}/temp/",
    "--enable-metrics": "true",
}

# YOUR CODE: Create the load job with glue.create_job(
#   Name=LOAD_JOB_NAME,
#   Role=role_arn,
#   Command={"Name": "glueetl", "ScriptLocation": load_script_uri, "PythonVersion": "3"},
#   DefaultArguments=load_default_args,
#   GlueVersion="4.0",
#   WorkerType="G.1X",
#   NumberOfWorkers=2,
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).

print(f"Load job created (bookmarks disabled by design): {LOAD_JOB_NAME}")

# Workflow. The workflow is the container; triggers are what actually
# chain the nodes together.
# YOUR CODE: Create the workflow with glue.create_workflow(
#   Name=WORKFLOW_NAME,
#   Description=f"Incremental Glue pipeline for {LAB_ID}",
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).

print(f"Workflow created: {WORKFLOW_NAME}")

# Three triggers wire the nodes together inside the workflow. Trigger 1
# is ON_DEMAND (the workflow starts when you call start_workflow_run).
# Triggers 2 and 3 are CONDITIONAL with State=SUCCEEDED predicates so the
# downstream step only fires on success of the upstream step.
# Both conditional triggers carry StartOnCreation=True so the workflow
# controller wakes them up the moment the workflow is created.

# YOUR CODE: Create the start trigger with glue.create_trigger(
#   Name=START_TRIGGER_NAME,
#   WorkflowName=WORKFLOW_NAME,
#   Type="ON_DEMAND",
#   Actions=[{"CrawlerName": CRAWLER_NAME}],
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).

# YOUR CODE: Create the transform trigger with glue.create_trigger(
#   Name=TRANSFORM_TRIGGER_NAME,
#   WorkflowName=WORKFLOW_NAME,
#   Type="CONDITIONAL",
#   Predicate={
#     "Logical": "ANY",
#     "Conditions": [{
#       "LogicalOperator": "EQUALS",
#       "CrawlerName": CRAWLER_NAME,
#       "CrawlState": "SUCCEEDED",
#     }],
#   },
#   Actions=[{"JobName": TRANSFORM_JOB_NAME}],
#   StartOnCreation=True,
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).

# YOUR CODE: Create the load trigger with glue.create_trigger(
#   Name=LOAD_TRIGGER_NAME,
#   WorkflowName=WORKFLOW_NAME,
#   Type="CONDITIONAL",
#   Predicate={
#     "Logical": "ANY",
#     "Conditions": [{
#       "LogicalOperator": "EQUALS",
#       "JobName": TRANSFORM_JOB_NAME,
#       "State": "SUCCEEDED",
#     }],
#   },
#   Actions=[{"JobName": LOAD_JOB_NAME}],
#   StartOnCreation=True,
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).

print(f"Three triggers created and wired: {START_TRIGGER_NAME}, {TRANSFORM_TRIGGER_NAME}, {LOAD_TRIGGER_NAME}")
print("Workflow scaffold is ready. Task 2 seeds files and starts the first run.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: workflow exists; graph has crawler + transform-job + load-job
# nodes; triggers in the workflow include one ON_DEMAND start, two
# CONDITIONAL triggers chained on SUCCEEDED.


def checkpoint_1(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            wf = glue_client.get_workflow(Name=WORKFLOW_NAME, IncludeGraph=True)
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Workflow {WORKFLOW_NAME} does not exist. Run the Task 1 cell.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        graph = wf["Workflow"].get("Graph", {})
        nodes = graph.get("Nodes", [])
        edges = graph.get("Edges", [])

        # Node-type inventory: CRAWLER, JOB, TRIGGER.
        crawler_nodes = [n for n in nodes if n.get("Type") == "CRAWLER"]
        job_nodes = [n for n in nodes if n.get("Type") == "JOB"]
        trigger_nodes = [n for n in nodes if n.get("Type") == "TRIGGER"]

        if not any(
            n.get("Name") == CRAWLER_NAME for n in crawler_nodes
        ):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Workflow graph is missing CRAWLER node {CRAWLER_NAME!r}. The "
                    f"start-trigger Actions list must reference the crawler by name."
                ),
            )
        job_names = {n.get("Name") for n in job_nodes}
        for required in (TRANSFORM_JOB_NAME, LOAD_JOB_NAME):
            if required not in job_names:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Workflow graph is missing JOB node {required!r}. The "
                        f"conditional triggers must include this job in their Actions."
                    ),
                )

        # Trigger inventory: must include all three lab triggers.
        trigger_names = {n.get("Name") for n in trigger_nodes}
        for required in (START_TRIGGER_NAME, TRANSFORM_TRIGGER_NAME, LOAD_TRIGGER_NAME):
            if required not in trigger_names:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Workflow graph is missing TRIGGER node {required!r}. Each "
                        f"trigger must be created with WorkflowName={WORKFLOW_NAME!r}."
                    ),
                )

        # Inspect each trigger's type and predicate via get_trigger.
        start = glue_client.get_trigger(Name=START_TRIGGER_NAME)
        start_type = start["Trigger"].get("Type")
        if start_type != "ON_DEMAND":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"start-trigger Type is {start_type!r}, expected ON_DEMAND. "
                    f"glue.start_workflow_run only fires triggers of type ON_DEMAND."
                ),
            )

        for trigger_name, expected_action_key in (
            (TRANSFORM_TRIGGER_NAME, "JobName"),
            (LOAD_TRIGGER_NAME, "JobName"),
        ):
            t = glue_client.get_trigger(Name=trigger_name)
            t_def = t["Trigger"]
            t_type = t_def.get("Type")
            if t_type != "CONDITIONAL":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{trigger_name} Type is {t_type!r}, expected CONDITIONAL. "
                        f"Conditional is what chains a job to fire on an upstream SUCCEEDED."
                    ),
                )
            predicate = t_def.get("Predicate", {})
            conditions = predicate.get("Conditions", [])
            if not conditions:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{trigger_name} Predicate.Conditions is empty. The trigger must "
                        f"name the upstream resource and the SUCCEEDED state explicitly."
                    ),
                )
            states = {c.get("State") or c.get("CrawlState") for c in conditions}
            if "SUCCEEDED" not in states:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{trigger_name} Predicate.Conditions do not require SUCCEEDED. "
                        f"Without the state filter, the downstream step would fire on FAILED too."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Checkpoint 1 walks four things: the workflow exists; the graph has a CRAWLER node named `labrun-glue-workflows-and-bookmarks-crawler`; the graph has JOB nodes for the transform and load jobs; the start trigger is `ON_DEMAND` and the two downstream triggers are `CONDITIONAL` with `State=SUCCEEDED` (or `CrawlState=SUCCEEDED` on the after-crawler trigger). The most common miss is leaving `StartOnCreation=False` on the conditional triggers; without it the workflow controller never wires the trigger into the graph and the graph walk sees the trigger but not the JOB it would have fired.

</details>

<details><summary>Hint 2 (direction)</summary>

Ten API calls in this task: two `s3.put_object` (transform script, load script), `iam.create_role`, `iam.put_role_policy`, `glue.create_crawler`, two `glue.create_job` (transform with bookmark enabled, load without), `glue.create_workflow`, three `glue.create_trigger` (one ON_DEMAND start, two CONDITIONAL). The 15-second IAM propagation sleep is already in the cell between the role attach and the crawler/job create. The two conditional triggers must pass `StartOnCreation=True` so the workflow controller actually evaluates them at runtime.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1 (the script bodies, trust policy, and inline policy are already defined as constants in the cell):

```python
s3.create_bucket(Bucket=BUCKET_NAME)
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

s3.put_object(
    Bucket=BUCKET_NAME,
    Key=TRANSFORM_SCRIPT_KEY,
    Body=TRANSFORM_SCRIPT.encode("utf-8"),
)
s3.put_object(
    Bucket=BUCKET_NAME,
    Key=LOAD_SCRIPT_KEY,
    Body=LOAD_SCRIPT.encode("utf-8"),
)

iam.create_role(
    RoleName=ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(inline_policy),
)

glue.create_crawler(
    Name=CRAWLER_NAME,
    Role=role_arn,
    DatabaseName=DATABASE_NAME,
    Targets={"S3Targets": [{"Path": f"s3://{BUCKET_NAME}/staging/"}]},
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

glue.create_job(
    Name=TRANSFORM_JOB_NAME,
    Role=role_arn,
    Command={"Name": "glueetl", "ScriptLocation": transform_script_uri, "PythonVersion": "3"},
    DefaultArguments=transform_default_args,
    GlueVersion="4.0",
    WorkerType="G.1X",
    NumberOfWorkers=2,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
glue.create_job(
    Name=LOAD_JOB_NAME,
    Role=role_arn,
    Command={"Name": "glueetl", "ScriptLocation": load_script_uri, "PythonVersion": "3"},
    DefaultArguments=load_default_args,
    GlueVersion="4.0",
    WorkerType="G.1X",
    NumberOfWorkers=2,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

glue.create_workflow(
    Name=WORKFLOW_NAME,
    Description=f"Incremental Glue pipeline for {LAB_ID}",
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

glue.create_trigger(
    Name=START_TRIGGER_NAME,
    WorkflowName=WORKFLOW_NAME,
    Type="ON_DEMAND",
    Actions=[{"CrawlerName": CRAWLER_NAME}],
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
glue.create_trigger(
    Name=TRANSFORM_TRIGGER_NAME,
    WorkflowName=WORKFLOW_NAME,
    Type="CONDITIONAL",
    Predicate={
        "Logical": "ANY",
        "Conditions": [{
            "LogicalOperator": "EQUALS",
            "CrawlerName": CRAWLER_NAME,
            "CrawlState": "SUCCEEDED",
        }],
    },
    Actions=[{"JobName": TRANSFORM_JOB_NAME}],
    StartOnCreation=True,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
glue.create_trigger(
    Name=LOAD_TRIGGER_NAME,
    WorkflowName=WORKFLOW_NAME,
    Type="CONDITIONAL",
    Predicate={
        "Logical": "ANY",
        "Conditions": [{
            "LogicalOperator": "EQUALS",
            "JobName": TRANSFORM_JOB_NAME,
            "State": "SUCCEEDED",
        }],
    },
    Actions=[{"JobName": LOAD_JOB_NAME}],
    StartOnCreation=True,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
```

If `create_job` returns `InvalidInputException: The role ARN is not valid` and the role exists, the 15-second IAM propagation wait was not enough; sleep another 15 seconds and retry. If `create_trigger` returns `InvalidInputException: Predicate Condition must have CrawlState or State`, you missed the `CrawlState` key on the after-crawler trigger or the `State` key on the after-transform trigger; they look similar but Glue keys them differently.

</details>

**Wallet check.** Still essentially $0.00. Creating an IAM role, a Glue database, a crawler definition, two job definitions, a workflow, and three triggers is all free; you pay only when the workflow runs. The bucket is created and tagged; the two scripts (a few KB total) cost a fraction of a penny against the always-free S3 request budget. The morning coffee continues to win.

## Task 2: Seed three CSVs into staging/ and run the workflow once end to end

The workflow is built; nothing has run yet. This task seeds three deterministically-generated CSV files into `s3://{bucket}/staging/` and kicks the first workflow run.

What happens when `start_workflow_run` fires:
1. Workflow controller evaluates the `start-trigger` (ON_DEMAND); fires the crawler.
2. Crawler walks `staging/`, registers a table named `staging` in `labrun_glue_workflows_and_bookmarks_db`, finishes SUCCEEDED.
3. Workflow controller evaluates the `after-crawler` trigger; the crawler is in `SUCCEEDED`, so the transform job fires.
4. Transform job reads from the catalog via `from_catalog(... transformation_ctx="transform_orders_v1")`. Because no bookmark state exists yet, the job reads every staging file, derives `year`/`month`, drops duplicates on `order_id`, and writes Parquet under `curated/year=YYYY/month=MM/`. `job.commit()` persists the bookmark state.
5. Workflow controller evaluates the `after-transform` trigger; transform is in `SUCCEEDED`, so the load job fires and writes its run marker.

Total wall-clock: roughly 12 to 15 minutes. Most of it is the Glue 10-minute minimum on the transform job and the crawler's worker provisioning. The actual transform on 240 rows finishes in well under a minute. If a run lands in `FAILED`, the poll loop prints the CloudWatch log group name and the failed run id so you can grep the worker stderr.

Checkpoint 2 inspects the workflow run, asserts it reached `COMPLETED`, and lists `s3://{bucket}/curated/` to confirm at least one Parquet partition exists.

In [ ]:
# NBVAL_SKIP
# Task 2: Seed three CSVs into staging/, start workflow run 1, poll until
# the run reaches a terminal state, and stash the run id for Checkpoint 2.

random.seed(SEED)
COLUMNS = ["order_id", "customer_id", "amount_cents", "currency", "created_at"]
CURRENCIES = ["USD", "USD", "USD", "EUR", "GBP"]
BASE_EPOCH = 1730419200  # 2024-11-01T00:00:00Z
STEP_SECONDS = 86400  # 1 day per row so dates span multiple months


def _iso(offset_days: int) -> str:
    t = BASE_EPOCH + offset_days * STEP_SECONDS
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(t))


def _build_rows(prefix: str, start_offset: int, count: int) -> list[list]:
    rows = []
    for i in range(count):
        rows.append([
            f"ord_{prefix}_{i:04d}",
            f"cust_{random.randint(1000, 9999)}",
            random.randint(100, 99999),
            random.choice(CURRENCIES),
            _iso(start_offset + i),
        ])
    return rows


def _csv_bytes(rows: list[list]) -> bytes:
    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(COLUMNS)
    for r in rows:
        writer.writerow(r)
    return buf.getvalue().encode("utf-8")


alpha_rows = _build_rows("a", 0, ROWS_PER_FILE)
beta_rows = _build_rows("b", 30, ROWS_PER_FILE)
gamma_rows = _build_rows("g", 60, ROWS_PER_FILE)

STAGING_ALPHA_KEY = "staging/batch_001.csv"
STAGING_BETA_KEY = "staging/batch_002.csv"
STAGING_GAMMA_KEY = "staging/batch_003.csv"

# YOUR CODE: Upload the three CSVs to staging/ with
#   s3.put_object(Bucket=BUCKET_NAME, Key=STAGING_ALPHA_KEY, Body=_csv_bytes(alpha_rows))
# and the matching calls for beta and gamma.

print(f"Seeded 3 CSVs under s3://{BUCKET_NAME}/staging/")
print(f"  - {STAGING_ALPHA_KEY} ({ROWS_PER_FILE} rows)")
print(f"  - {STAGING_BETA_KEY} ({ROWS_PER_FILE} rows)")
print(f"  - {STAGING_GAMMA_KEY} ({ROWS_PER_FILE} rows)")

glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

WORKFLOW_TERMINAL_STATES = {"COMPLETED", "STOPPED", "ERROR"}

# YOUR CODE: Start the workflow with
#   run = glue.start_workflow_run(Name=WORKFLOW_NAME)
#   run_id_1 = run["RunId"]
# Note: glue.start_workflow_run requires WorkflowName=... on some SDK
# versions and Name=... on others. botocore accepts Name=...

print(f"Workflow run 1 submitted. RunId (set above) will be polled now.")
print("Crawler spins up, then transform, then load. Roughly 12 to 15 minutes.")

deadline = time.time() + 1500  # 25 minutes
last_state = None
run_resp_1 = None
while time.time() < deadline:
    run_resp_1 = glue.get_workflow_run(Name=WORKFLOW_NAME, RunId=run_id_1)
    state = run_resp_1["Run"]["Status"]
    if state != last_state:
        print(f"  state: {state}")
        last_state = state
    if state in WORKFLOW_TERMINAL_STATES:
        break
    time.sleep(20)
else:
    raise RuntimeError(
        f"Workflow run {run_id_1} did not reach a terminal state within 25 minutes. "
        f"Inspect the run in the AWS Glue console."
    )

final_state = run_resp_1["Run"]["Status"]
if final_state == "COMPLETED":
    stats = run_resp_1["Run"].get("Statistics", {})
    print(f"Workflow run 1 finished COMPLETED. Stats: {stats}")
else:
    print(f"Workflow run 1 finished {final_state}.")
    print(f"Inspect each node's run in the Glue console:")
    print(f"  Workflow: {WORKFLOW_NAME}, RunId: {run_id_1}")
    print("  CloudWatch logs: /aws-glue/jobs/output and /aws-glue/crawlers")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: workflow run 1 reached COMPLETED; curated/ now has at
# least one Parquet object under a year=YYYY/month=MM/ prefix.

PARQUET_KEY_RE = re.compile(
    r"^curated/year=(\d{4})/month=(\d{1,2})/[^/]+\.parquet$"
)


def checkpoint_2(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        if "run_id_1" not in globals():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "run_id_1 is not defined. The Task 2 cell must call "
                    "glue.start_workflow_run(Name=WORKFLOW_NAME) and assign "
                    'run_id_1 = run["RunId"] before this checkpoint runs.'
                ),
            )

        try:
            run = glue_client.get_workflow_run(Name=WORKFLOW_NAME, RunId=run_id_1)
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))

        status_val = run["Run"].get("Status")
        if status_val != "COMPLETED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Workflow run 1 status is {status_val!r}, expected COMPLETED. "
                    f"Wait for the workflow to finish, or inspect the failed node "
                    f"in the AWS Glue console for the workflow run page."
                ),
            )

        # Curated/ should have at least one Parquet object under year/month.
        paginator = s3_client.get_paginator("list_objects_v2")
        parquet_keys: list[str] = []
        partition_set: set[tuple[str, str]] = set()
        for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="curated/"):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                base = key.rsplit("/", 1)[-1]
                if base.startswith("_") or base.startswith("."):
                    continue
                m = PARQUET_KEY_RE.match(key)
                if not m:
                    continue
                parquet_keys.append(key)
                partition_set.add((m.group(1), m.group(2)))
        if not parquet_keys:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No Parquet objects found under curated/ year=YYYY/month=MM/. "
                    "Confirm the transform job ran and writes "
                    "df.write.partitionBy('year', 'month').parquet to "
                    "s3://{bucket}/curated/."
                ),
            )
        if len(partition_set) < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Found {len(partition_set)} distinct (year, month) partitions; "
                    f"expected at least 1. The seed spans roughly 4 months so this "
                    f"should never trip."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Checkpoint 2 reads three things: `run_id_1` exists in the cell scope, `get_workflow_run` reports `Status=COMPLETED`, and at least one Parquet object lives under `curated/year=YYYY/month=MM/`. If the run status is `ERROR`, click into the workflow run page in the AWS Glue console to see which node failed; the most common first-run failure is the crawler not finding `staging/` (empty bucket) or the transform job failing on a missing table because the crawler did not actually run (start trigger not wired). If the run is `COMPLETED` but no Parquet exists, the transform script's `from_catalog` got an empty DynamicFrame (likely the crawler missed the staging table) and the script's early-exit branch fired.

</details>

<details><summary>Hint 2 (direction)</summary>

Two student lines in this cell: three `s3.put_object` uploads to seed the staging files, and one `glue.start_workflow_run(Name=WORKFLOW_NAME)` call that captures `run_id_1 = run["RunId"]`. The poll loop is already written. The workflow controller takes care of firing the downstream triggers once the upstream node reaches `SUCCEEDED`; you do not need to call `start_crawler` or `start_job_run` directly. If you want to inspect a specific node's run within the workflow, use `glue.get_workflow_run(Name=..., RunId=..., IncludeGraph=True)` and walk `Run.Graph.Nodes[].JobDetails` or `CrawlerDetails`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2 (the poll loop is unchanged):

```python
s3.put_object(Bucket=BUCKET_NAME, Key=STAGING_ALPHA_KEY, Body=_csv_bytes(alpha_rows))
s3.put_object(Bucket=BUCKET_NAME, Key=STAGING_BETA_KEY, Body=_csv_bytes(beta_rows))
s3.put_object(Bucket=BUCKET_NAME, Key=STAGING_GAMMA_KEY, Body=_csv_bytes(gamma_rows))

run = glue.start_workflow_run(Name=WORKFLOW_NAME)
run_id_1 = run["RunId"]
```

If `start_workflow_run` returns `InvalidInputException: The workflow has no run-on-demand triggers`, the start trigger was created with the wrong `Type` (anything other than `ON_DEMAND`) or without `WorkflowName=WORKFLOW_NAME`. If the workflow status reaches `COMPLETED` but the curated bucket is empty, the transform job hit its early-exit branch (zero rows read); the most likely cause is the crawler's table name is not `staging` (the prefix-derived name). Pass `STAGING_TABLE` explicitly via the transform job's `DefaultArguments` instead of inferring it.

</details>

**Wallet check.** This is the first cell that materially moves the bill. The workflow runs three nodes: one crawler (2 DPU, 1-minute minimum at $0.44/DPU-hour: about $0.015), one transform job (2 G.1X workers, 10-minute minimum at $0.44/DPU-hour: about $0.15), one load job (same as transform: about $0.15). Total for run 1: around $0.30. Storage and request costs are well under a cent. Coffee is still ahead of us, but only just.

## Task 3: Run the workflow a second time with no new staging files and prove the bookmark held

The bookmark on the transform job was committed at the end of run 1. The catalog already lists the staging table. Nothing has changed in `staging/`. What should happen on a second `start_workflow_run`:

1. Crawler runs again; finds no new files; finishes `SUCCEEDED` (crawlers are not bookmarked but they handle re-runs of the same data gracefully).
2. Transform job fires. The bookmark state says "the three batch files were already processed." `from_catalog(... transformation_ctx="transform_orders_v1")` returns an empty DynamicFrame. The script's early-exit branch fires, `job.commit()` runs (which moves the bookmark watermark forward to the same point it already was), and the job lands `SUCCEEDED`.
3. Load job fires (it always fires when transform succeeds); writes another run marker. The marker is fine; we are counting Parquet partitions, not markers.

Checkpoint 3 inspects two things: workflow run 2 reached `COMPLETED`, and the count of Parquet objects under `curated/` is exactly the same as it was after run 1. If the bookmark did not hold, the transform job would write a second copy of the three batches and the count would double; the checkpoint catches that and prints the bookmark configuration to grep for.

This is the most exam-relevant checkpoint in the lab. DEA-C01 questions on Glue 4.0 bookmarks usually frame it as: "the script re-processes the same files on every run; what is missing?" The answer is either the `--job-bookmark-option: job-bookmark-enable` argument on the job, or the stable `transformation_ctx` on the `from_catalog` call (or both).

In [ ]:
# NBVAL_SKIP
# Task 3: Start workflow run 2 with no new staging files. The bookmark on
# the transform job should keep the transform from reprocessing the same
# three files, leaving the curated/ object count unchanged.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Snapshot curated/ Parquet object count BEFORE run 2.


def _count_curated_parquet_objects() -> int:
    paginator = s3.get_paginator("list_objects_v2")
    count = 0
    for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="curated/"):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            base = key.rsplit("/", 1)[-1]
            if base.startswith("_") or base.startswith("."):
                continue
            if not key.endswith(".parquet"):
                continue
            count += 1
    return count


PARQUET_COUNT_BEFORE_RUN_2 = _count_curated_parquet_objects()
print(f"Curated parquet object count BEFORE run 2: {PARQUET_COUNT_BEFORE_RUN_2}")

WORKFLOW_TERMINAL_STATES = {"COMPLETED", "STOPPED", "ERROR"}

# YOUR CODE: Start the workflow a second time with
#   run = glue.start_workflow_run(Name=WORKFLOW_NAME)
#   run_id_2 = run["RunId"]

print("Workflow run 2 submitted (no new files since run 1).")
print("Bookmark should hold: transform reads zero rows, writes nothing.")

deadline = time.time() + 1500  # 25 minutes
last_state = None
run_resp_2 = None
while time.time() < deadline:
    run_resp_2 = glue.get_workflow_run(Name=WORKFLOW_NAME, RunId=run_id_2)
    state = run_resp_2["Run"]["Status"]
    if state != last_state:
        print(f"  state: {state}")
        last_state = state
    if state in WORKFLOW_TERMINAL_STATES:
        break
    time.sleep(20)
else:
    raise RuntimeError(
        f"Workflow run {run_id_2} did not reach a terminal state within 25 minutes."
    )

PARQUET_COUNT_AFTER_RUN_2 = _count_curated_parquet_objects()
print(f"Curated parquet object count AFTER run 2: {PARQUET_COUNT_AFTER_RUN_2}")
delta = PARQUET_COUNT_AFTER_RUN_2 - PARQUET_COUNT_BEFORE_RUN_2
print(f"Parquet object count delta from run 2: {delta}")
if delta == 0:
    print("Bookmark worked. Transform job processed zero new files.")
else:
    print(f"Bookmark did NOT hold. Run 2 produced {delta} new Parquet object(s).")
    print("Inspect the transform job's DefaultArguments for --job-bookmark-option")
    print("and the PySpark script for a stable transformation_ctx on from_catalog.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: workflow run 2 reached COMPLETED and the curated/ Parquet
# object count is unchanged between run 1 and run 2 (bookmark held).


def checkpoint_3(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        if "run_id_2" not in globals():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "run_id_2 is not defined. The Task 3 cell must call "
                    "glue.start_workflow_run a second time and assign "
                    'run_id_2 = run["RunId"] before this checkpoint runs.'
                ),
            )

        run = glue_client.get_workflow_run(Name=WORKFLOW_NAME, RunId=run_id_2)
        status_val = run["Run"].get("Status")
        if status_val != "COMPLETED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Workflow run 2 status is {status_val!r}, expected COMPLETED. "
                    f"Wait for the workflow to finish or inspect the failed node "
                    f"in the AWS Glue console."
                ),
            )

        # Confirm the bookmark held: parquet object count unchanged.
        if "PARQUET_COUNT_BEFORE_RUN_2" not in globals() or "PARQUET_COUNT_AFTER_RUN_2" not in globals():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "PARQUET_COUNT_BEFORE_RUN_2 / PARQUET_COUNT_AFTER_RUN_2 are "
                    "not set. The Task 3 cell snapshots these around the second "
                    "workflow run; rerun the cell before this checkpoint."
                ),
            )
        delta = PARQUET_COUNT_AFTER_RUN_2 - PARQUET_COUNT_BEFORE_RUN_2
        if delta != 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Curated parquet object count grew by {delta} on run 2; "
                    f"expected 0 because no new staging files were added. The "
                    f"bookmark did not hold. Check the transform job's DefaultArguments "
                    f"for '--job-bookmark-option': 'job-bookmark-enable' and the "
                    f"PySpark script for a stable transformation_ctx on from_catalog."
                ),
            )

        # Cross-check: the transform job's DefaultArguments still say bookmark-enable.
        job_def = glue_client.get_job(JobName=TRANSFORM_JOB_NAME)
        args = job_def["Job"].get("DefaultArguments", {})
        bookmark_opt = args.get("--job-bookmark-option")
        if bookmark_opt != "job-bookmark-enable":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Transform job --job-bookmark-option is {bookmark_opt!r}, "
                    f"expected 'job-bookmark-enable'. Without it, the Glue "
                    f"framework does not persist bookmark state across runs."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Checkpoint 3 reads two signals: the curated Parquet object count is unchanged across run 2, and the transform job's `DefaultArguments` still includes `--job-bookmark-option: job-bookmark-enable`. If the count grew (delta > 0), the transform reprocessed the same files; either the bookmark argument is missing or the script's `transformation_ctx` changed between runs (a different string per run defeats the bookmark). If the argument is missing entirely, `glue.update_job` can fix it in place without recreating the job.

</details>

<details><summary>Hint 2 (direction)</summary>

One student line in this cell: `run = glue.start_workflow_run(Name=WORKFLOW_NAME)` and `run_id_2 = run["RunId"]`. The snapshot calls and poll loop are pre-written. If the run lands in `ERROR` with the transform node failing on "table not found," the crawler did not actually run (the start trigger is misconfigured) or the table got dropped between runs; re-run the workflow from Task 2 first.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3 (the rest of the cell is pre-written):

```python
run = glue.start_workflow_run(Name=WORKFLOW_NAME)
run_id_2 = run["RunId"]
```

If `glue.update_job` is needed to fix the bookmark argument after the fact, the pattern is `glue.update_job(JobName=TRANSFORM_JOB_NAME, JobUpdate={"Role": role_arn, "Command": {...}, "DefaultArguments": {..., "--job-bookmark-option": "job-bookmark-enable"}})`. `JobUpdate` must include every field you want to keep, not just the field you change. If the bookmark argument is correct but the bookmark still does not hold, the script's `transformation_ctx` is the culprit; it must be a stable string literal across runs (do not derive it from the current time, the run id, or the file name).

</details>

**Wallet check.** Run 2 added another ~$0.30 (one crawler at $0.015, one transform job at $0.15 even though it processed zero rows because the 10-minute minimum still applies, one load job at $0.15). Cumulative session spend: about $0.60. The transform job billed the same as run 1 even though it did zero useful work; that is the cost of the Glue 10-minute minimum, and it is the reason bookmarks are most useful when the alternative is rescanning hundreds of GB of input on every run (where the savings are measured in dollars per run, not cents).

## Task 4: Add one new staging CSV, run the workflow a third time, and confirm exactly one new partition

The last move proves the incremental shape of the pipeline. Upload one fresh CSV under `staging/batch_004.csv` covering a new month, kick `start_workflow_run` once more, and confirm:

1. Workflow run 3 reached `COMPLETED`.
2. Curated Parquet object count grew by at least one between run 2 and run 3.
3. The new objects sit under a partition `(year, month)` value that did not exist before run 3.
4. The curated dataset has no duplicate `order_id` values; the new file's rows joined the curated set without colliding.

Why this is the harshest checkpoint of the four: a misconfigured bookmark would let the transform job rescan the original three files AND process the new one, doubling some partitions. A misconfigured partition write would scatter the new file across existing partitions. A missing dedup step would leave duplicate `order_id` values from the new file overlapping with existing data. The checkpoint walks all three failure modes.

In [ ]:
# NBVAL_SKIP
# Task 4: Upload one new CSV that covers a new month into staging/,
# start workflow run 3, and confirm exactly one new partition appears in
# curated/.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Snapshot partition set BEFORE run 3.


def _list_curated_partitions() -> set[tuple[str, str]]:
    paginator = s3.get_paginator("list_objects_v2")
    partitions: set[tuple[str, str]] = set()
    pattern = re.compile(r"^curated/year=(\d{4})/month=(\d{1,2})/")
    for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="curated/"):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            m = pattern.match(key)
            if m:
                partitions.add((m.group(1), m.group(2)))
    return partitions


PARTITIONS_BEFORE_RUN_3 = _list_curated_partitions()
print(f"Curated partitions BEFORE run 3: {sorted(PARTITIONS_BEFORE_RUN_3)}")

# Build one new CSV covering a new month (start offset 120 days lands the
# rows in a month past the previous three batches).
random.seed(SEED + 1)
new_rows = []
for i in range(NEW_FILE_ROWS):
    new_rows.append([
        f"ord_new_{i:04d}",
        f"cust_{random.randint(1000, 9999)}",
        random.randint(100, 99999),
        random.choice(CURRENCIES),
        _iso(120 + i),
    ])

new_csv_bytes = _csv_bytes(new_rows)
STAGING_NEW_KEY = "staging/batch_004.csv"

# YOUR CODE: Upload the new CSV to staging/ with
#   s3.put_object(Bucket=BUCKET_NAME, Key=STAGING_NEW_KEY, Body=new_csv_bytes)

print(f"Uploaded new file: {STAGING_NEW_KEY} ({NEW_FILE_ROWS} rows in a new month)")

WORKFLOW_TERMINAL_STATES = {"COMPLETED", "STOPPED", "ERROR"}

# YOUR CODE: Start the workflow a third time with
#   run = glue.start_workflow_run(Name=WORKFLOW_NAME)
#   run_id_3 = run["RunId"]

print("Workflow run 3 submitted. Bookmark should process ONLY the new file.")

deadline = time.time() + 1500
last_state = None
run_resp_3 = None
while time.time() < deadline:
    run_resp_3 = glue.get_workflow_run(Name=WORKFLOW_NAME, RunId=run_id_3)
    state = run_resp_3["Run"]["Status"]
    if state != last_state:
        print(f"  state: {state}")
        last_state = state
    if state in WORKFLOW_TERMINAL_STATES:
        break
    time.sleep(20)
else:
    raise RuntimeError(
        f"Workflow run {run_id_3} did not reach a terminal state within 25 minutes."
    )

PARTITIONS_AFTER_RUN_3 = _list_curated_partitions()
print(f"Curated partitions AFTER run 3: {sorted(PARTITIONS_AFTER_RUN_3)}")
new_partitions = PARTITIONS_AFTER_RUN_3 - PARTITIONS_BEFORE_RUN_3
print(f"New partitions introduced by run 3: {sorted(new_partitions)}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: workflow run 3 reached COMPLETED; curated/ has exactly one
# new partition compared to run 2; no duplicate order_id values across the
# full curated dataset.


def checkpoint_4(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        if "run_id_3" not in globals():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "run_id_3 is not defined. The Task 4 cell must call "
                    "glue.start_workflow_run a third time and assign "
                    'run_id_3 = run["RunId"] before this checkpoint runs.'
                ),
            )

        run = glue_client.get_workflow_run(Name=WORKFLOW_NAME, RunId=run_id_3)
        status_val = run["Run"].get("Status")
        if status_val != "COMPLETED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Workflow run 3 status is {status_val!r}, expected COMPLETED."
                ),
            )

        # Exactly one new partition.
        if "PARTITIONS_BEFORE_RUN_3" not in globals() or "PARTITIONS_AFTER_RUN_3" not in globals():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "PARTITIONS_BEFORE_RUN_3 / PARTITIONS_AFTER_RUN_3 are not set. "
                    "The Task 4 cell snapshots these around the third workflow run."
                ),
            )
        new_parts = PARTITIONS_AFTER_RUN_3 - PARTITIONS_BEFORE_RUN_3
        if len(new_parts) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Run 3 introduced {len(new_parts)} new partition(s): "
                    f"{sorted(new_parts)}. Expected exactly 1. The new staging file "
                    f"covers a single new month; the transform job should append rows "
                    f"to exactly one (year, month) partition. If 0, the bookmark is too "
                    f"aggressive (or the file did not land under staging/). If >1, the "
                    f"new rows span months unexpectedly; check the _iso offset."
                ),
            )

        # No duplicate order_id values in the curated dataset.
        try:
            import pyarrow.parquet as pq  # noqa: F401
            from io import BytesIO
        except ImportError:
            return CheckpointResult(
                status="error",
                error_reason=(
                    "pyarrow is not installed in this Colab runtime. Run "
                    "!pip install pyarrow and re-run this checkpoint."
                ),
            )

        # Walk every Parquet object and count distinct order_id.
        paginator = s3_client.get_paginator("list_objects_v2")
        seen_order_ids: set = set()
        total_rows = 0
        duplicate_examples: list[str] = []
        for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="curated/"):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                base = key.rsplit("/", 1)[-1]
                if base.startswith("_") or base.startswith(".") or not key.endswith(".parquet"):
                    continue
                got = s3_client.get_object(Bucket=BUCKET_NAME, Key=key)
                buf = BytesIO(got["Body"].read())
                table = pq.read_table(buf)
                if table.num_rows == 0:
                    continue
                cols = table.column_names
                if "order_id" not in cols:
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"Parquet file {key} is missing the order_id column. "
                            f"The transform script must keep order_id in the write set."
                        ),
                    )
                for oid in table.column("order_id").to_pylist():
                    if oid in seen_order_ids:
                        if len(duplicate_examples) < 3:
                            duplicate_examples.append(oid)
                    seen_order_ids.add(oid)
                total_rows += table.num_rows

        if duplicate_examples:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Found duplicate order_id values across the curated dataset: "
                    f"{duplicate_examples}. The transform job must call "
                    f"dropDuplicates(['order_id']) before write. If the bookmark "
                    f"held but duplicates still appear, the dedup is missing."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Checkpoint 4 walks three things: run 3 finished `COMPLETED`, exactly one new `(year, month)` partition appeared between snapshots, and no duplicate `order_id` exists across the full curated dataset (read via pyarrow). If 0 new partitions appeared, the bookmark dropped the new file (most likely the `transformation_ctx` value changed between Task 2 and Task 4) or the file did not land under `staging/`. If more than 1, the new file spans more months than expected. If duplicates exist, the script's `dropDuplicates(['order_id'])` is missing.

</details>

<details><summary>Hint 2 (direction)</summary>

Two student lines: `s3.put_object(Bucket=BUCKET_NAME, Key=STAGING_NEW_KEY, Body=new_csv_bytes)` and `run = glue.start_workflow_run(Name=WORKFLOW_NAME); run_id_3 = run["RunId"]`. The snapshots and poll loop are pre-written. The new CSV's row offsets (`_iso(120 + i)`) place the rows in a month past the original seed range, which is the only way to land exactly one new partition.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4 (the rest is pre-written):

```python
s3.put_object(Bucket=BUCKET_NAME, Key=STAGING_NEW_KEY, Body=new_csv_bytes)

run = glue.start_workflow_run(Name=WORKFLOW_NAME)
run_id_3 = run["RunId"]
```

If exactly zero new partitions show up after a `COMPLETED` run, walk the bookmark plumbing in this order: confirm the new CSV is under `staging/` (not `staging-new/` or `raw/`), confirm the crawler re-ran and the `staging` table still resolves (`glue.get_table(DatabaseName=DATABASE_NAME, Name=STAGING_TABLE_NAME)` returns), confirm the transform job's `--job-bookmark-option` is still `job-bookmark-enable`, and confirm the PySpark script's `transformation_ctx` is the same literal it was on run 1.

</details>

**Wallet check.** Run 3 added another ~$0.30 (crawler + transform + load). Cumulative session spend: about $0.90 if everything went smoothly on the first try, more if you reran the workflow after a script tweak. The cleanup cell below deletes every job, trigger, workflow, crawler, role, database, and bucket so the meter stops the moment you finish.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. Lab 07 has no critical (hourly-billed) resources.
#
# labrun-checks 0.3.0 ships AWS adapters for s3_bucket, iam_role,
# glue_database, and glue_table. It does NOT yet support glue_job,
# glue_crawler, glue_workflow, or glue_trigger. A _LabAdapter subclass
# wraps AwsCleanupAdapter and adds those four resource types as inline
# fallbacks. When the package promotes them in a future release, the
# wrapper can be removed and run_cleanup called against the default.

import sys
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    """AwsCleanupAdapter wrapper that adds glue_job, glue_crawler,
    glue_workflow, and glue_trigger support. Triggers must be deactivated
    before delete on some Glue versions; we do that explicitly here.
    """

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def _glue(self, credentials: dict):
        return boto3.client(
            "glue",
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "glue_job":
            client = self._glue(credentials)
            try:
                client.delete_job(JobName=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return
                raise
            return
        if resource.type == "glue_crawler":
            client = self._glue(credentials)
            # Stop the crawler first if it is RUNNING or STOPPING; delete is
            # rejected while a crawl is in flight.
            try:
                state = client.get_crawler(Name=resource.id)
                cstate = state["Crawler"].get("State")
                if cstate in ("RUNNING", "STOPPING"):
                    try:
                        client.stop_crawler(Name=resource.id)
                    except ClientError:
                        pass
                    # Wait briefly for crawler to actually stop.
                    for _ in range(12):
                        time.sleep(5)
                        cur = client.get_crawler(Name=resource.id)
                        if cur["Crawler"].get("State") == "READY":
                            break
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return
            try:
                client.delete_crawler(Name=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return
                raise
            return
        if resource.type == "glue_workflow":
            client = self._glue(credentials)
            try:
                client.delete_workflow(Name=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return
                raise
            return
        if resource.type == "glue_trigger":
            client = self._glue(credentials)
            # Some Glue versions require triggers to be DEACTIVATED before
            # delete. stop_trigger is idempotent; swallow the no-op.
            try:
                client.stop_trigger(Name=resource.id)
            except ClientError as e:
                code = e.response["Error"]["Code"]
                if code in ("EntityNotFoundException", "ConcurrentRunsExceededException"):
                    pass
                elif code in (
                    "InvalidInputException",
                    "EntityNotFoundException",
                ):
                    pass
            try:
                client.delete_trigger(Name=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return
                raise
            return
        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "glue_job":
            client = self._glue(credentials)
            try:
                client.get_job(JobName=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise
        if resource.type == "glue_crawler":
            client = self._glue(credentials)
            try:
                client.get_crawler(Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise
        if resource.type == "glue_workflow":
            client = self._glue(credentials)
            try:
                client.get_workflow(Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise
        if resource.type == "glue_trigger":
            client = self._glue(credentials)
            try:
                client.get_trigger(Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise
        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before run_cleanup tears it down. S3 buckets cannot be
# deleted while they contain objects, and the workflow wrote raw CSVs, two
# Glue scripts, curated Parquet under multiple partitions, the load
# marker, and Glue temp shuffle output under /temp/.
print("Emptying the S3 bucket before teardown...")
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.30 to $0.75.** Glue ETL job runs and crawler runs are the only line items that materially register. Three workflow runs each spin up one crawler (about 1.5 cents), one transform job (about 15 cents at the 10-minute minimum), and one load job (about 15 cents at the 10-minute minimum), so each workflow run is roughly 30 cents. Run 1 processed real data; run 2 only proved the bookmark held; run 3 picked up the single new file. If you redrove the workflow after a script tweak you may have landed closer to a dollar. Everything else (S3 storage and requests, IAM, Glue Data Catalog, CloudWatch Logs) is rounding error against the DPU-hours. The cleanup cell above deleted every resource so the bill stops accruing now.

## These are not graded. They are for you.

Three questions worth sitting with before the exam.

1. The transform job's bookmark held because two things lined up: `--job-bookmark-option: job-bookmark-enable` was set on the job, and the PySpark script created the source DynamicFrame with a stable `transformation_ctx`. Walk through what happens if you keep the job argument but change the `transformation_ctx` literal between deployments (for example, append a build version number to the string each release). Then walk through what happens if you keep the `transformation_ctx` stable but flip the job argument to `job-bookmark-pause`. Both are subtle bugs the DEA-C01 exam likes to test.

2. This lab used a Glue workflow with three triggers to express the dependency graph. Identify two situations where you would prefer Step Functions over a Glue workflow for the same kind of orchestration, and two where the Glue workflow is the right call. The cost angle matters (Step Functions bills per state transition; Glue workflows are free). The error-handling angle matters more (Step Functions has retry and catch primitives that Glue workflows do not).

3. The load step in this lab was deliberately trivial: it writes a marker file. In a real pipeline that load step would push the curated Parquet to Redshift via COPY, or register partitions in the Glue catalog, or refresh a Lake Formation tag. Lab 06 (`redshift-loading-and-queries`) walks the Redshift COPY pattern. If you swap the load script for a real Redshift COPY call, do you want bookmarks enabled on the load step or not? The answer is not the same as the transform step.

## Exam-Style Practice

**Q1.** A data engineer enables `--job-bookmark-option: job-bookmark-enable` on a Glue 4.0 ETL job. The job reads from a Glue catalog table via `from_catalog` and writes Parquet to S3. On the second run with no new source files, the job reprocesses every row again. Which single change is most likely to fix the bookmark?

A) Switch the source from `from_catalog` to `from_options` reading the S3 path directly.

B) Add a stable `transformation_ctx` argument to the `create_dynamic_frame.from_catalog` call.

C) Increase `NumberOfWorkers` from 2 to 4 so the bookmark state has more shards to write to.

D) Replace `job-bookmark-enable` with `job-bookmark-pause` so the framework stops re-reading the bookmark each run.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. `from_options` supports bookmarks too, but only if `transformation_ctx` is set; the missing key is `transformation_ctx`, not the source method.
- B) Right. The Glue bookmark state is keyed on the `transformation_ctx` string. Without it (or with a different value across runs), the framework cannot match the read to a prior bookmark watermark, so it re-reads everything.
- C) Wrong. Worker count controls parallelism and memory; the bookmark state is a small JSON document keyed per `(job, transformation_ctx)` pair, unrelated to shard count.
- D) Wrong. `job-bookmark-pause` reads the bookmark on the way in but does not advance it on commit; that makes the problem worse, not better.

</details>

**Q2.** A Glue workflow chains a crawler to a transform job using a single conditional trigger. The trigger's predicate is `[{"LogicalOperator": "EQUALS", "CrawlerName": "my-crawler"}]` with no `CrawlState` field. On the first workflow run the crawler fails and the transform job fires anyway, then crashes because the catalog table does not exist. What is the cleanest fix?

A) Add a separate try/except around the transform script to handle missing tables.

B) Change the predicate to add `"CrawlState": "SUCCEEDED"` so the trigger only fires when the crawler reaches SUCCEEDED.

C) Replace the conditional trigger with an ON_DEMAND trigger and start the transform job manually after each crawler run.

D) Wrap the trigger in a Step Functions state machine so failure handling is centralized.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Catching the exception in the script papers over the workflow misconfiguration; the upstream failure should not cascade into the downstream node in the first place.
- B) Right. A conditional trigger without a state filter fires on any terminal state of the upstream node, including `FAILED` and `STOPPED`. Adding `"CrawlState": "SUCCEEDED"` is the canonical fix and the DEA-C01 exam tests this specific pattern.
- C) Wrong. Replacing the conditional with ON_DEMAND defeats the purpose of the workflow; manual coordination is what the workflow exists to avoid.
- D) Wrong. Step Functions is a valid orchestrator but the question is about fixing the trigger predicate, not switching tools.

</details>

**Q3.** A data engineer needs to confirm a Glue 4.0 bookmark is actually advancing after each successful run. Glue 4.0 does not expose bookmark state via the API. Which inspection path is the most reliable and the closest to what production teams use?

A) Open the AWS Glue console and read the Bookmark State column on the job run page.

B) Have the PySpark script print the read-side row count and a stable observation marker to stdout on each run, then grep the CloudWatch Logs Insights query results for the marker.

C) Query the `aws_glue` system catalog table from Athena with `SELECT * FROM bookmark_state WHERE job_name = ...`.

D) Call `glue.get_job_bookmark(JobName=..., RunId=...)` from boto3 and read the `LastUpdatedOn` field.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. The Glue console does not expose a bookmark state column for 4.0 jobs; it shows job run state, not the bookmark watermark.
- B) Right. The Glue 4.0 framework persists bookmark state in an internal store that is not exposed via API. The canonical pattern is to make the script print observable markers (row count, partition list, bookmark-state literal) to stdout, then read CloudWatch Logs Insights against `/aws-glue/jobs/output` to confirm. This lab's transform script prints exactly that pattern.
- C) Wrong. There is no `aws_glue.bookmark_state` Athena table; bookmarks are not exposed via the data catalog.
- D) Wrong. boto3 has no `get_job_bookmark` method on Glue 4.0; the `reset_job_bookmark` method exists but only resets state, it does not read it.

</details>